# Putting RNNs to Work -- Real Sequence Tasks

Welcome back! Over the past four notebooks we built the tools:

- **Notebook 01**: Vanilla RNN -- the foundation of sequence processing
- **Notebook 02**: LSTM -- long-term memory with gates
- **Notebook 03**: GRU -- a simpler gated architecture
- **Notebook 04**: Training tricks (gradient clipping, teacher forcing, etc.)

Now it is time to **use** those building blocks on real-world sequence tasks. In this notebook we will implement three classic applications from scratch:

| # | Task | Pattern | Output |
|---|------|---------|--------|
| 1 | **Character-Level Text Generation** | Many-to-Many | Generate new text one character at a time |
| 2 | **Sentiment Analysis** | Many-to-One | Classify a sentence as positive or negative |
| 3 | **Time Series Prediction** | Many-to-One (sliding window) | Predict the next value in a sequence |

All three use the **same** RNN/LSTM/GRU building blocks -- just wired differently. By the end you will see that the core idea (process a sequence step by step, accumulate hidden state, produce output) is incredibly versatile.

**Prerequisites:** Notebooks 01-04 (understanding of RNN, LSTM, and GRU cells).

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

np.random.seed(42)

print("Libraries imported successfully!")
print("NumPy version:", np.__version__)

---

## Helper Functions and Building Blocks

For self-containedness we redefine every helper function and RNN cell we will need. If you have already gone through notebooks 01-04, these will look familiar.

In [ ]:
# ===========================================================
# Helper functions
# ===========================================================

def sigmoid(x):
    """Numerically stable sigmoid."""
    x = np.clip(x, -500, 500)
    return 1.0 / (1.0 + np.exp(-x))


def softmax(x):
    """Numerically stable softmax over the last axis."""
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e / np.sum(e, axis=-1, keepdims=True)


def cross_entropy_loss(probs, target_idx):
    """Cross-entropy loss for a single sample.
    probs: probability vector (after softmax)
    target_idx: integer index of the correct class
    """
    return -np.log(probs[target_idx] + 1e-12)


def binary_cross_entropy(y_pred, y_true):
    """Binary cross-entropy loss."""
    y_pred = np.clip(y_pred, 1e-12, 1 - 1e-12)
    return -(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))


def mse_loss(y_pred, y_true):
    """Mean squared error loss."""
    return np.mean((y_pred - y_true) ** 2)


print("Helper functions defined: sigmoid, softmax, cross_entropy_loss, binary_cross_entropy, mse_loss")

In [ ]:
# ===========================================================
# LSTM Cell (from Notebook 02, re-implemented here)
# ===========================================================

class LSTMCell:
    """A single LSTM cell with forget, input, and output gates.
    
    Equations:
        f_t = sigmoid(W_f @ [h_{t-1}, x_t] + b_f)   -- forget gate
        i_t = sigmoid(W_i @ [h_{t-1}, x_t] + b_i)   -- input gate
        c_tilde = tanh(W_c @ [h_{t-1}, x_t] + b_c)  -- candidate cell
        c_t = f_t * c_{t-1} + i_t * c_tilde          -- new cell state
        o_t = sigmoid(W_o @ [h_{t-1}, x_t] + b_o)    -- output gate
        h_t = o_t * tanh(c_t)                         -- new hidden state
    """

    def __init__(self, input_size, hidden_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        concat_size = hidden_size + input_size
        scale = np.sqrt(2.0 / concat_size)

        # Weights for the four gates (forget, input, candidate, output)
        self.W_f = np.random.randn(hidden_size, concat_size) * scale
        self.b_f = np.ones((hidden_size, 1)) * 1.0   # bias toward remembering

        self.W_i = np.random.randn(hidden_size, concat_size) * scale
        self.b_i = np.zeros((hidden_size, 1))

        self.W_c = np.random.randn(hidden_size, concat_size) * scale
        self.b_c = np.zeros((hidden_size, 1))

        self.W_o = np.random.randn(hidden_size, concat_size) * scale
        self.b_o = np.zeros((hidden_size, 1))

    def forward_step(self, x_t, h_prev, c_prev):
        """One time step.
        
        Parameters:
            x_t:    (input_size, 1)
            h_prev: (hidden_size, 1)
            c_prev: (hidden_size, 1)
        Returns:
            h_t, c_t
        """
        concat = np.vstack([h_prev, x_t])  # (hidden_size + input_size, 1)

        f_t = sigmoid(self.W_f @ concat + self.b_f)
        i_t = sigmoid(self.W_i @ concat + self.b_i)
        c_tilde = np.tanh(self.W_c @ concat + self.b_c)
        c_t = f_t * c_prev + i_t * c_tilde
        o_t = sigmoid(self.W_o @ concat + self.b_o)
        h_t = o_t * np.tanh(c_t)

        return h_t, c_t


print("LSTMCell class defined.")

In [ ]:
# ===========================================================
# Vanilla RNN Cell (from Notebook 01)
# ===========================================================

class RNNCell:
    """Vanilla RNN cell: h_t = tanh(W_xh @ x_t + W_hh @ h_{t-1} + b_h)"""

    def __init__(self, input_size, hidden_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        scale = np.sqrt(2.0 / (input_size + hidden_size))
        self.W_xh = np.random.randn(hidden_size, input_size) * scale
        self.W_hh = np.random.randn(hidden_size, hidden_size) * scale
        self.b_h = np.zeros((hidden_size, 1))

    def forward_step(self, x_t, h_prev):
        h_t = np.tanh(self.W_xh @ x_t + self.W_hh @ h_prev + self.b_h)
        return h_t


# ===========================================================
# GRU Cell (from Notebook 03)
# ===========================================================

class GRUCell:
    """GRU cell with reset and update gates.
    
    z_t = sigmoid(W_z @ [h_{t-1}, x_t] + b_z)       -- update gate
    r_t = sigmoid(W_r @ [h_{t-1}, x_t] + b_r)       -- reset gate
    h_tilde = tanh(W_h @ [r_t * h_{t-1}, x_t] + b_h)-- candidate
    h_t = (1 - z_t) * h_{t-1} + z_t * h_tilde       -- new state
    """

    def __init__(self, input_size, hidden_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        concat_size = hidden_size + input_size
        scale = np.sqrt(2.0 / concat_size)

        self.W_z = np.random.randn(hidden_size, concat_size) * scale
        self.b_z = np.zeros((hidden_size, 1))

        self.W_r = np.random.randn(hidden_size, concat_size) * scale
        self.b_r = np.zeros((hidden_size, 1))

        self.W_h = np.random.randn(hidden_size, concat_size) * scale
        self.b_h = np.zeros((hidden_size, 1))

    def forward_step(self, x_t, h_prev):
        concat = np.vstack([h_prev, x_t])
        z_t = sigmoid(self.W_z @ concat + self.b_z)
        r_t = sigmoid(self.W_r @ concat + self.b_r)
        concat_reset = np.vstack([r_t * h_prev, x_t])
        h_tilde = np.tanh(self.W_h @ concat_reset + self.b_h)
        h_t = (1 - z_t) * h_prev + z_t * h_tilde
        return h_t


print("RNNCell and GRUCell classes defined.")

---

## Task 1: Character-Level Text Generation

### The Idea

Character-level text generation is one of the most fun things you can do with an RNN. The concept is simple:

1. **Train** the network to predict the *next* character given all the characters seen so far.
2. **Generate** new text by feeding in a seed character, sampling the next one from the output distribution, and repeating.

Think of it like **autocomplete on your phone**, but instead of predicting the next *word* it predicts the next *character*. The model learns spelling, word boundaries, and even simple grammar -- all by itself, just from raw characters.

### The Architecture (Many-to-Many)

```
Input:   h  e  l  l  o  _  w  o  r  l
                  |  |  |  |  |  |  |  |
         LSTM -> LSTM -> LSTM -> LSTM -> ...
                  |  |  |  |  |  |  |  |
Target:  e  l  l  o  _  w  o  r  l  d
```

At every time step the model outputs a probability distribution over all characters. We train it so that the correct next character has the highest probability.

In [ ]:
# ===========================================================
# Step 1: Prepare the data
# ===========================================================

# A small inline text corpus.  We intentionally keep it tiny so that
# training runs in a few seconds with pure NumPy.
text = (
    "the cat sat on the mat. "
    "the dog sat on the log. "
    "the cat and the dog sat on the mat. "
    "a cat is not a dog. "
    "the mat is on the log. "
    "the dog and the cat sat on the log. "
    "a dog sat on a mat. "
    "the cat sat and the dog sat. "
)

# Build character vocabulary
chars = sorted(set(text))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}

print(f"Text length: {len(text)} characters")
print(f"Unique characters: {vocab_size}")
print(f"Vocabulary: {chars}")
print()

# Encode text as integer indices
data = np.array([char_to_idx[ch] for ch in text])
print(f"Encoded text (first 40 chars): {data[:40]}")
print(f"Decoded back: {''.join(idx_to_char[i] for i in data[:40])}")

In [ ]:
# ===========================================================
# Step 2: Create training sequences
# ===========================================================
# We chop the text into overlapping windows of length seq_len.
# Input:  characters 0..seq_len-1
# Target: characters 1..seq_len   (shifted by one)

seq_len = 25  # number of characters per training example

inputs_list = []
targets_list = []

for i in range(0, len(data) - seq_len, seq_len):
    inputs_list.append(data[i : i + seq_len])
    targets_list.append(data[i + 1 : i + seq_len + 1])

print(f"Number of training sequences: {len(inputs_list)}")
print(f"Sequence length: {seq_len}")
print()

# Show one example
example_in = inputs_list[0]
example_tgt = targets_list[0]
print("Example input : ", ''.join(idx_to_char[i] for i in example_in))
print("Example target: ", ''.join(idx_to_char[i] for i in example_tgt))
print()
print("Notice: the target is the input shifted by one position.")
print("The model must learn to predict the next character at every step.")

In [ ]:
# ===========================================================
# Step 3: Build the character-level LSTM model
# ===========================================================

class CharLSTM:
    """Character-level language model using a single LSTM layer.
    
    Architecture:
        one-hot input -> LSTM cell -> linear layer -> softmax -> next char
    """

    def __init__(self, vocab_size, hidden_size, lr=0.01):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.lr = lr

        # LSTM cell
        self.lstm = LSTMCell(vocab_size, hidden_size)

        # Output projection: hidden -> vocab
        scale = np.sqrt(2.0 / hidden_size)
        self.W_out = np.random.randn(vocab_size, hidden_size) * scale
        self.b_out = np.zeros((vocab_size, 1))

    def forward(self, inputs, h_prev, c_prev):
        """Forward pass over a sequence of character indices.
        
        Returns:
            loss:  scalar, average cross-entropy over all time steps
            probs_list: list of probability vectors (for gradient computation)
            h, c:  final hidden and cell states
            cache: everything needed for backprop
        """
        xs, hs, cs, probs_list = {}, {}, {}, {}
        hs[-1] = h_prev.copy()
        cs[-1] = c_prev.copy()
        loss = 0.0

        for t in range(len(inputs)):
            # One-hot encode the input character
            xs[t] = np.zeros((self.vocab_size, 1))
            xs[t][inputs[t]] = 1.0

            # LSTM step
            hs[t], cs[t] = self.lstm.forward_step(xs[t], hs[t - 1], cs[t - 1])

            # Output logits and probabilities
            logits = self.W_out @ hs[t] + self.b_out
            probs_list[t] = softmax(logits.flatten())

        return probs_list, hs, cs

    def compute_loss(self, probs_list, targets):
        """Compute average cross-entropy loss."""
        loss = 0.0
        for t in range(len(targets)):
            loss += cross_entropy_loss(probs_list[t], targets[t])
        return loss / len(targets)

    def train_step_numerical(self, inputs, targets, h_prev, c_prev, epsilon=1e-5):
        """One training step using numerical gradients.
        
        This is slow but correct and easy to understand.
        In practice you would use backpropagation through time (BPTT).
        Here we use it for simplicity.
        """
        # We update only W_out and b_out (the output projection)
        # and the LSTM weights via a simple finite-difference approach.
        # For efficiency we only numerically differentiate the output layer
        # and do an approximate LSTM update.

        # --- Forward pass ---
        probs_list, hs, cs = self.forward(inputs, h_prev, c_prev)
        loss = self.compute_loss(probs_list, targets)

        # --- Gradient for output layer (analytical, simple) ---
        dW_out = np.zeros_like(self.W_out)
        db_out = np.zeros_like(self.b_out)

        for t in range(len(targets)):
            dy = probs_list[t].copy().reshape(-1, 1)
            dy[targets[t]] -= 1.0  # softmax gradient: p - 1(y=k)
            dW_out += dy @ hs[t].T
            db_out += dy

        dW_out /= len(targets)
        db_out /= len(targets)

        # Gradient clipping
        for grad in [dW_out, db_out]:
            np.clip(grad, -5, 5, out=grad)

        self.W_out -= self.lr * dW_out
        self.b_out -= self.lr * db_out

        # --- Approximate LSTM weight update ---
        # We use the error signal from the output layer to nudge LSTM weights.
        # This is a simplified update (not full BPTT) but works for our demo.
        for t in range(len(targets)):
            dy = probs_list[t].copy().reshape(-1, 1)
            dy[targets[t]] -= 1.0
            dh = self.W_out.T @ dy  # gradient flowing into hidden state

            # One-hot input
            x_t = np.zeros((self.vocab_size, 1))
            x_t[inputs[t]] = 1.0
            concat = np.vstack([hs[t - 1], x_t])

            # Nudge each LSTM gate weight slightly in the direction that reduces loss
            scale_factor = self.lr * 0.1 / len(targets)
            self.lstm.W_f -= scale_factor * np.clip(dh @ concat.T, -5, 5)
            self.lstm.W_i -= scale_factor * np.clip(dh @ concat.T, -5, 5)
            self.lstm.W_c -= scale_factor * np.clip(dh @ concat.T, -5, 5)
            self.lstm.W_o -= scale_factor * np.clip(dh @ concat.T, -5, 5)

        h_final = hs[len(inputs) - 1]
        c_final = cs[len(inputs) - 1]
        return loss, h_final, c_final

    def generate(self, seed_char_idx, length, temperature=1.0):
        """Generate text character by character.
        
        Parameters:
            seed_char_idx: index of the starting character
            length: how many characters to generate
            temperature: controls randomness (low = conservative, high = creative)
        """
        h = np.zeros((self.hidden_size, 1))
        c = np.zeros((self.hidden_size, 1))
        idx = seed_char_idx
        generated = [idx]

        for _ in range(length):
            x = np.zeros((self.vocab_size, 1))
            x[idx] = 1.0
            h, c = self.lstm.forward_step(x, h, c)
            logits = (self.W_out @ h + self.b_out).flatten()

            # Apply temperature
            logits = logits / temperature
            probs = softmax(logits)

            # Sample from the distribution
            idx = np.random.choice(len(probs), p=probs)
            generated.append(idx)

        return ''.join(idx_to_char[i] for i in generated)


print("CharLSTM model defined.")
print("  - Uses an LSTM cell to process characters one by one")
print("  - Output layer projects hidden state to vocabulary probabilities")
print("  - generate() method samples new text character by character")

In [ ]:
# ===========================================================
# Step 4: Train the character model
# ===========================================================

hidden_size = 64
model = CharLSTM(vocab_size, hidden_size, lr=0.05)

num_epochs = 200
loss_history = []
samples_at_stages = {}  # save generated text at different training stages

print("Training character-level LSTM...")
print("=" * 60)

for epoch in range(num_epochs):
    epoch_loss = 0.0
    h = np.zeros((hidden_size, 1))
    c = np.zeros((hidden_size, 1))

    for inputs_seq, targets_seq in zip(inputs_list, targets_list):
        loss, h, c = model.train_step_numerical(inputs_seq, targets_seq, h, c)
        epoch_loss += loss

    avg_loss = epoch_loss / len(inputs_list)
    loss_history.append(avg_loss)

    # Save samples at certain epochs
    if epoch in [0, 19, 49, 99, 149, 199]:
        sample = model.generate(char_to_idx['t'], 60, temperature=0.8)
        samples_at_stages[epoch] = sample
        print(f"Epoch {epoch+1:3d}/{num_epochs}  Loss: {avg_loss:.4f}  Sample: '{sample[:50]}...'")

print("=" * 60)
print("Training complete!")

In [ ]:
# ===========================================================
# Step 5: Visualize training progress
# ===========================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss curve
axes[0].plot(loss_history, color='#2980B9', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Average Loss', fontsize=12)
axes[0].set_title('Training Loss Curve', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Mark the epochs where we saved samples
for ep in samples_at_stages:
    axes[0].axvline(x=ep, color='red', linestyle='--', alpha=0.4)
    axes[0].annotate(f'Epoch {ep+1}', xy=(ep, loss_history[ep]),
                     xytext=(ep + 5, loss_history[ep] + 0.1),
                     fontsize=8, color='red')

# Generated text at different stages
axes[1].axis('off')
axes[1].set_title('Generated Text at Different Stages', fontsize=14, fontweight='bold')
y_pos = 0.95
for ep, sample in samples_at_stages.items():
    axes[1].text(0.02, y_pos, f'Epoch {ep+1:3d}:', fontsize=10, fontweight='bold',
                 transform=axes[1].transAxes, verticalalignment='top',
                 fontfamily='monospace')
    axes[1].text(0.18, y_pos, f'"{sample[:45]}"', fontsize=9,
                 transform=axes[1].transAxes, verticalalignment='top',
                 fontfamily='monospace', color='#2980B9')
    y_pos -= 0.15

plt.tight_layout()
plt.show()

print("\nNotice how the generated text improves:")
print("  - Early epochs: random gibberish")
print("  - Middle epochs: starts to look like words")
print("  - Late epochs: recognizable patterns from the training text")

In [ ]:
# ===========================================================
# Step 6: Generate text at different temperatures
# ===========================================================

print("TEXT GENERATION WITH DIFFERENT TEMPERATURES")
print("=" * 60)
print()
print("Temperature controls how 'adventurous' the model is:")
print("  Low  (0.2) = very conservative, repetitive but safe")
print("  Med  (0.8) = balanced between variety and correctness")
print("  High (1.5) = very creative, but often makes mistakes")
print()

temperatures = [0.2, 0.5, 0.8, 1.0, 1.5]
seed_idx = char_to_idx['t']

for temp in temperatures:
    np.random.seed(42)  # same seed for fair comparison
    generated = model.generate(seed_idx, 80, temperature=temp)
    print(f"  Temperature {temp:.1f}: \"{generated}\"")
    print()

print("=" * 60)
print("\nKey takeaway: temperature is a knob between predictability and creativity.")
print("In practice, values around 0.7-1.0 usually give the best results.")

---

## Task 2: Sentiment Analysis (Many-to-One)

### The Idea

Sentiment analysis is about reading a piece of text and deciding: **is this positive or negative?**

Think of it like reading a movie review. After reading the whole thing, you give a thumbs up or thumbs down. The RNN does the same thing:

1. Read the sentence **word by word**, updating the hidden state at each step.
2. After the last word, use the **final hidden state** to make a single prediction.

This is the **many-to-one** pattern:

```
Input:   great  movie  really  enjoyed  it
            |      |       |       |      |
          LSTM -> LSTM -> LSTM -> LSTM -> LSTM
                                           |
                                       Positive!
```

Only the **last** hidden state produces an output. All the earlier hidden states are just building up context.

In [ ]:
# ===========================================================
# Step 1: Create a synthetic sentiment dataset
# ===========================================================

# We use simple keyword-based sentences.
# In the real world you would use actual movie reviews, but this
# lets us train quickly and see the pattern clearly.

positive_sentences = [
    "great movie really enjoyed",
    "wonderful film loved it",
    "amazing story great acting",
    "fantastic film really great",
    "loved the wonderful story",
    "great acting amazing film",
    "really enjoyed wonderful movie",
    "amazing great wonderful loved",
    "enjoyed the great story",
    "wonderful great loved it",
    "great film enjoyed it",
    "loved amazing wonderful movie",
    "fantastic wonderful great film",
    "really loved the movie",
    "amazing story enjoyed it",
]

negative_sentences = [
    "terrible movie really hated",
    "awful film boring story",
    "bad acting terrible film",
    "horrible movie really bad",
    "hated the awful story",
    "bad acting horrible film",
    "really hated awful movie",
    "terrible bad awful hated",
    "boring the bad story",
    "awful bad hated it",
    "terrible film hated it",
    "hated horrible awful movie",
    "boring awful terrible film",
    "really hated the movie",
    "bad story hated it",
]

# Combine and label: 1 = positive, 0 = negative
sentences = positive_sentences + negative_sentences
labels = [1] * len(positive_sentences) + [0] * len(negative_sentences)

# Shuffle
shuffle_idx = np.random.permutation(len(sentences))
sentences = [sentences[i] for i in shuffle_idx]
labels = [labels[i] for i in shuffle_idx]

print(f"Dataset: {len(sentences)} sentences ({len(positive_sentences)} positive, {len(negative_sentences)} negative)")
print()
print("Sample data:")
for i in range(5):
    sentiment = "Positive" if labels[i] == 1 else "Negative"
    print(f"  [{sentiment:8s}] \"{sentences[i]}\"")

In [ ]:
# ===========================================================
# Step 2: Build word vocabulary and encode
# ===========================================================

# Collect all unique words
all_words = set()
for s in sentences:
    for w in s.split():
        all_words.add(w)

word_to_idx = {w: i for i, w in enumerate(sorted(all_words))}
idx_to_word = {i: w for w, i in word_to_idx.items()}
sentiment_vocab_size = len(word_to_idx)

print(f"Vocabulary size: {sentiment_vocab_size} words")
print(f"Words: {sorted(all_words)}")
print()


def encode_sentence(sentence, word_to_idx):
    """Convert a sentence string to a list of word indices."""
    return [word_to_idx[w] for w in sentence.split()]


# Encode all sentences
encoded_sentences = [encode_sentence(s, word_to_idx) for s in sentences]

print("Example encoding:")
print(f"  Sentence: \"{sentences[0]}\"")
print(f"  Encoded:  {encoded_sentences[0]}")

In [ ]:
# ===========================================================
# Step 3: Build the many-to-one sentiment classifier
# ===========================================================

class SentimentLSTM:
    """Many-to-one LSTM classifier for binary sentiment analysis.
    
    Architecture:
        one-hot word -> LSTM (all steps) -> take LAST hidden state
        -> linear layer -> sigmoid -> probability of positive
    """

    def __init__(self, vocab_size, hidden_size, lr=0.05):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.lr = lr

        self.lstm = LSTMCell(vocab_size, hidden_size)

        # Output layer: hidden -> 1 (binary)
        scale = np.sqrt(2.0 / hidden_size)
        self.W_out = np.random.randn(1, hidden_size) * scale
        self.b_out = np.zeros((1, 1))

    def forward(self, word_indices):
        """Process an entire sentence, return prediction from the last hidden state.
        
        Returns:
            prob: probability of positive (scalar)
            all_hidden: list of hidden states at every step (for visualization)
        """
        h = np.zeros((self.hidden_size, 1))
        c = np.zeros((self.hidden_size, 1))
        all_hidden = []

        for idx in word_indices:
            x = np.zeros((self.vocab_size, 1))
            x[idx] = 1.0
            h, c = self.lstm.forward_step(x, h, c)
            all_hidden.append(h.copy())

        # Use ONLY the last hidden state for classification
        logit = self.W_out @ h + self.b_out
        prob = sigmoid(logit.item())

        return prob, all_hidden

    def train_step(self, word_indices, label):
        """One training step with simplified gradient update."""
        prob, all_hidden = self.forward(word_indices)
        loss = binary_cross_entropy(prob, label)

        # Gradient of BCE + sigmoid: (prob - label)
        d_logit = prob - label

        # Update output layer
        h_last = all_hidden[-1]
        dW_out = d_logit * h_last.T
        db_out = d_logit

        np.clip(dW_out, -5, 5, out=dW_out)
        self.W_out -= self.lr * dW_out
        self.b_out -= self.lr * np.array([[db_out]])

        # Approximate LSTM update (simplified)
        dh = (self.W_out.T * d_logit)  # gradient into last hidden state

        h_prev = np.zeros((self.hidden_size, 1))
        for t, idx in enumerate(word_indices):
            x = np.zeros((self.vocab_size, 1))
            x[idx] = 1.0
            concat = np.vstack([h_prev, x])
            update = self.lr * 0.05 * np.clip(dh @ concat.T, -5, 5)
            self.lstm.W_f -= update
            self.lstm.W_i -= update
            self.lstm.W_c -= update
            self.lstm.W_o -= update
            h_prev = all_hidden[t]

        return loss, prob


print("SentimentLSTM model defined.")
print("  - Reads the whole sentence word by word")
print("  - Uses only the LAST hidden state for classification")
print("  - Sigmoid output: probability of being positive")

In [ ]:
# ===========================================================
# Step 4: Train the sentiment classifier
# ===========================================================

np.random.seed(42)
sentiment_model = SentimentLSTM(sentiment_vocab_size, hidden_size=32, lr=0.05)

num_epochs_sent = 100
loss_history_sent = []
acc_history_sent = []

print("Training sentiment classifier...")
print("=" * 60)

for epoch in range(num_epochs_sent):
    epoch_loss = 0.0
    correct = 0

    # Shuffle training data each epoch
    perm = np.random.permutation(len(sentences))

    for i in perm:
        loss, prob = sentiment_model.train_step(encoded_sentences[i], labels[i])
        epoch_loss += loss
        pred = 1 if prob > 0.5 else 0
        if pred == labels[i]:
            correct += 1

    avg_loss = epoch_loss / len(sentences)
    accuracy = correct / len(sentences)
    loss_history_sent.append(avg_loss)
    acc_history_sent.append(accuracy)

    if (epoch + 1) % 20 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:3d}/{num_epochs_sent}  Loss: {avg_loss:.4f}  Accuracy: {accuracy:.1%}")

print("=" * 60)
print(f"Final accuracy: {acc_history_sent[-1]:.1%}")

In [ ]:
# ===========================================================
# Step 5: Visualize training and test on new sentences
# ===========================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(loss_history_sent, color='#E74C3C', linewidth=2, label='Loss')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Sentiment Classifier -- Training Loss', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=11)

# Accuracy curve
axes[1].plot(acc_history_sent, color='#27AE60', linewidth=2, label='Accuracy')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Sentiment Classifier -- Training Accuracy', fontsize=14, fontweight='bold')
axes[1].set_ylim(0, 1.05)
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# ===========================================================
# Step 6: Test on new sentences
# ===========================================================

test_sentences = [
    "great movie loved it",
    "terrible film hated it",
    "wonderful story amazing acting",
    "awful boring bad movie",
    "really enjoyed great film",
    "horrible terrible awful film",
]

print("TESTING ON NEW SENTENCES")
print("=" * 60)

for sentence in test_sentences:
    encoded = encode_sentence(sentence, word_to_idx)
    prob, _ = sentiment_model.forward(encoded)
    pred = "Positive" if prob > 0.5 else "Negative"
    confidence = prob if prob > 0.5 else 1 - prob
    print(f"  \"{sentence}\"")
    print(f"    -> {pred} (confidence: {confidence:.1%})")
    print()

print("=" * 60)

In [ ]:
# ===========================================================
# Step 7: Visualize which words activate the hidden state most
# ===========================================================

def visualize_hidden_activation(model, sentence, word_to_idx, idx_to_word):
    """Show how the hidden state changes as the model reads each word.
    
    We measure the L2 norm of the change in hidden state after each word.
    Larger changes mean the word had a bigger impact on the model's thinking.
    """
    encoded = encode_sentence(sentence, word_to_idx)
    prob, all_hidden = model.forward(encoded)

    words = sentence.split()
    changes = []

    for t in range(len(all_hidden)):
        if t == 0:
            prev = np.zeros_like(all_hidden[0])
        else:
            prev = all_hidden[t - 1]
        change = np.linalg.norm(all_hidden[t] - prev)
        changes.append(change)

    return words, changes, prob


# Visualize two example sentences
fig, axes = plt.subplots(2, 1, figsize=(12, 7))

examples = [
    "great movie really enjoyed",
    "terrible film really hated",
]
colors_list = ['#27AE60', '#E74C3C']

for ax_idx, (sentence, color) in enumerate(zip(examples, colors_list)):
    words, changes, prob = visualize_hidden_activation(
        sentiment_model, sentence, word_to_idx, idx_to_word
    )
    pred = "Positive" if prob > 0.5 else "Negative"

    bars = axes[ax_idx].bar(range(len(words)), changes, color=color, alpha=0.7, edgecolor='black')
    axes[ax_idx].set_xticks(range(len(words)))
    axes[ax_idx].set_xticklabels(words, fontsize=13, fontweight='bold')
    axes[ax_idx].set_ylabel('Hidden State Change (L2)', fontsize=11)
    axes[ax_idx].set_title(
        f'\"{sentence}\" -> {pred} ({prob:.1%})',
        fontsize=13, fontweight='bold'
    )
    axes[ax_idx].grid(axis='y', alpha=0.3)

plt.suptitle('Which Words Impact the Hidden State Most?', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  Taller bars = the word caused a bigger change in the hidden state.")
print("  Sentiment words (great, terrible, enjoyed, hated) typically cause")
print("  larger changes than neutral words (movie, film, really).")

---

## Task 3: Time Series Prediction

### The Idea

Time series prediction is about looking at past values and predicting what comes next. Think of it like **predicting tomorrow's temperature based on the past week**.

The approach:
1. **Sliding window**: take the last $k$ values as input.
2. **Feed them through an RNN** one step at a time.
3. Use the final hidden state to predict the **next** value.

This is another many-to-one pattern, but now the output is a continuous number (regression) instead of a class.

```
Input:   y_{t-4}  y_{t-3}  y_{t-2}  y_{t-1}
            |        |        |        |
          RNN  ->  RNN  ->  RNN  ->  RNN
                                       |
                                    y_t (predicted)
```

In [ ]:
# ===========================================================
# Step 1: Generate a synthetic time series
# ===========================================================

np.random.seed(42)

# Combination of sine waves with noise -- like a simplified real signal
t = np.linspace(0, 8 * np.pi, 500)
signal = 0.6 * np.sin(t) + 0.3 * np.sin(2.5 * t) + 0.1 * np.sin(5 * t)
noise = np.random.randn(len(t)) * 0.05
time_series = signal + noise

# Train/test split
train_size = 400
train_data = time_series[:train_size]
test_data = time_series[train_size:]

# Visualize
plt.figure(figsize=(14, 4))
plt.plot(range(train_size), train_data, color='#2980B9', label='Train', linewidth=1.5)
plt.plot(range(train_size, len(time_series)), test_data, color='#E74C3C', label='Test', linewidth=1.5)
plt.axvline(x=train_size, color='black', linestyle='--', alpha=0.5, label='Train/Test split')
plt.xlabel('Time step', fontsize=12)
plt.ylabel('Value', fontsize=12)
plt.title('Synthetic Time Series (sum of sine waves + noise)', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Total data points: {len(time_series)}")
print(f"Training points:   {train_size}")
print(f"Test points:       {len(test_data)}")

In [ ]:
# ===========================================================
# Step 2: Create sliding window input/output pairs
# ===========================================================

def create_windows(data, window_size):
    """Create sliding window input/output pairs.
    
    Input:  [y_{t-k}, y_{t-k+1}, ..., y_{t-1}]  (window_size values)
    Output: y_t  (the next value)
    """
    X, Y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i : i + window_size])
        Y.append(data[i + window_size])
    return np.array(X), np.array(Y)


window_size = 20
X_train, Y_train = create_windows(train_data, window_size)
X_test, Y_test = create_windows(test_data, window_size)

print(f"Window size: {window_size}")
print(f"Training samples: {len(X_train)} windows")
print(f"Test samples:     {len(X_test)} windows")
print()
print(f"Example window (input):  {X_train[0][:5]}... ({window_size} values)")
print(f"Example target (output): {Y_train[0]:.4f} (next value)")

In [ ]:
# ===========================================================
# Step 3: Build a simple RNN for time series prediction
# ===========================================================

class TimeSeriesRNN:
    """Simple RNN for predicting the next value in a time series.
    
    Architecture:
        scalar input -> RNN cell (all steps) -> last hidden state
        -> linear layer (no activation!) -> predicted value
    
    No activation on the output because this is regression
    (we want to predict any real number, not just 0-1).
    """

    def __init__(self, cell_type, input_size, hidden_size, lr=0.001):
        """
        Parameters:
            cell_type: 'rnn', 'lstm', or 'gru'
            input_size: dimension of each input (1 for univariate time series)
            hidden_size: size of the hidden state
            lr: learning rate
        """
        self.cell_type = cell_type
        self.hidden_size = hidden_size
        self.lr = lr

        if cell_type == 'rnn':
            self.cell = RNNCell(input_size, hidden_size)
        elif cell_type == 'lstm':
            self.cell = LSTMCell(input_size, hidden_size)
        elif cell_type == 'gru':
            self.cell = GRUCell(input_size, hidden_size)
        else:
            raise ValueError(f"Unknown cell type: {cell_type}")

        # Output layer: hidden -> 1 (predicted value)
        scale = np.sqrt(2.0 / hidden_size)
        self.W_out = np.random.randn(1, hidden_size) * scale
        self.b_out = np.zeros((1, 1))

    def forward(self, window):
        """Process a window of values and predict the next one.
        
        Parameters:
            window: array of shape (window_size,)
        Returns:
            prediction: scalar
            all_hidden: list of hidden states
        """
        h = np.zeros((self.hidden_size, 1))
        c = np.zeros((self.hidden_size, 1)) if self.cell_type == 'lstm' else None
        all_hidden = []

        for val in window:
            x = np.array([[val]])  # shape (1, 1)
            if self.cell_type == 'lstm':
                h, c = self.cell.forward_step(x, h, c)
            else:
                h = self.cell.forward_step(x, h)
            all_hidden.append(h.copy())

        # Linear output (no activation -- regression!)
        prediction = (self.W_out @ h + self.b_out).item()
        return prediction, all_hidden

    def train_step(self, window, target):
        """One training step with simplified gradient update."""
        pred, all_hidden = self.forward(window)
        loss = (pred - target) ** 2  # MSE for single sample

        # Gradient of MSE: 2 * (pred - target)
        d_pred = 2.0 * (pred - target)

        # Update output layer
        h_last = all_hidden[-1]
        dW_out = d_pred * h_last.T
        db_out = np.array([[d_pred]])

        np.clip(dW_out, -5, 5, out=dW_out)
        self.W_out -= self.lr * dW_out
        self.b_out -= self.lr * np.clip(db_out, -5, 5)

        # Approximate RNN weight update
        dh = self.W_out.T * d_pred
        h_prev = np.zeros((self.hidden_size, 1))

        for t_idx, val in enumerate(window):
            x = np.array([[val]])

            if self.cell_type == 'rnn':
                grad_xh = np.clip(dh @ x.T, -5, 5) * self.lr * 0.01
                grad_hh = np.clip(dh @ h_prev.T, -5, 5) * self.lr * 0.01
                self.cell.W_xh -= grad_xh
                self.cell.W_hh -= grad_hh
            elif self.cell_type == 'lstm':
                concat = np.vstack([h_prev, x])
                update = self.lr * 0.01 * np.clip(dh @ concat.T, -5, 5)
                self.cell.W_f -= update
                self.cell.W_i -= update
                self.cell.W_c -= update
                self.cell.W_o -= update
            elif self.cell_type == 'gru':
                concat = np.vstack([h_prev, x])
                update = self.lr * 0.01 * np.clip(dh @ concat.T, -5, 5)
                self.cell.W_z -= update
                self.cell.W_r -= update
                self.cell.W_h -= update

            h_prev = all_hidden[t_idx]

        return loss


print("TimeSeriesRNN model defined.")
print("  - Supports 'rnn', 'lstm', and 'gru' cell types")
print("  - Takes a window of past values, predicts the next value")
print("  - Linear output (no activation) for regression")

In [ ]:
# ===========================================================
# Step 4: Train the time series model
# ===========================================================

np.random.seed(42)
ts_model = TimeSeriesRNN('lstm', input_size=1, hidden_size=16, lr=0.005)

num_epochs_ts = 30
loss_history_ts = []

print("Training time series LSTM...")
print("=" * 60)

for epoch in range(num_epochs_ts):
    epoch_loss = 0.0
    perm = np.random.permutation(len(X_train))

    for i in perm:
        loss = ts_model.train_step(X_train[i], Y_train[i])
        epoch_loss += loss

    avg_loss = epoch_loss / len(X_train)
    loss_history_ts.append(avg_loss)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:2d}/{num_epochs_ts}  MSE: {avg_loss:.6f}")

print("=" * 60)
print("Training complete!")

In [ ]:
# ===========================================================
# Step 5: Visualize predictions vs actual
# ===========================================================

# Predict on test set
predictions_test = []
for i in range(len(X_test)):
    pred, _ = ts_model.forward(X_test[i])
    predictions_test.append(pred)
predictions_test = np.array(predictions_test)

# Also predict on training set for comparison
predictions_train = []
for i in range(len(X_train)):
    pred, _ = ts_model.forward(X_train[i])
    predictions_train.append(pred)
predictions_train = np.array(predictions_train)

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# Plot 1: Full view
train_x = np.arange(window_size, train_size)
test_x = np.arange(train_size + window_size, train_size + window_size + len(predictions_test))

axes[0].plot(range(len(time_series)), time_series, color='#2980B9', alpha=0.5, linewidth=1, label='Actual')
axes[0].plot(train_x, predictions_train, color='#27AE60', linewidth=1.5, label='Predicted (train)', alpha=0.7)
axes[0].plot(test_x, predictions_test, color='#E74C3C', linewidth=1.5, label='Predicted (test)')
axes[0].axvline(x=train_size, color='black', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Time step', fontsize=12)
axes[0].set_ylabel('Value', fontsize=12)
axes[0].set_title('Time Series Prediction -- Full View', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Plot 2: Zoomed into test set
axes[1].plot(test_x, Y_test, color='#2980B9', linewidth=2, label='Actual', marker='o', markersize=3)
axes[1].plot(test_x, predictions_test, color='#E74C3C', linewidth=2, label='Predicted', marker='s', markersize=3)
axes[1].fill_between(test_x, Y_test, predictions_test, alpha=0.2, color='gray', label='Error')
axes[1].set_xlabel('Time step', fontsize=12)
axes[1].set_ylabel('Value', fontsize=12)
axes[1].set_title('Time Series Prediction -- Test Set (zoomed)', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

test_mse = mse_loss(predictions_test, Y_test)
print(f"Test MSE: {test_mse:.6f}")
print()
print("Observations:")
print("  - The model learns the overall wave pattern quite well.")
print("  - It may struggle at peaks and sharp changes (high-frequency components).")
print("  - The gray shaded area shows where predictions deviate from actual values.")

---

## Comparing Architectures on the Same Task

Now for the real test: let us train all three architectures (vanilla RNN, LSTM, GRU) on the **same** time series task and compare them head-to-head.

This lets us see:
- Which one **converges fastest**?
- Which one achieves the **lowest error**?
- How do they handle the **long-range patterns** in the signal?

In [ ]:
# ===========================================================
# Train all three architectures and compare
# ===========================================================

architectures = {
    'Vanilla RNN': 'rnn',
    'LSTM': 'lstm',
    'GRU': 'gru',
}

results = {}  # will store loss histories and predictions
num_epochs_compare = 30
hidden_size_compare = 16

print("Training all three architectures on the same time series...")
print("=" * 60)

for name, cell_type in architectures.items():
    np.random.seed(42)  # same initialization for fair comparison
    model = TimeSeriesRNN(cell_type, input_size=1, hidden_size=hidden_size_compare, lr=0.005)

    losses = []
    for epoch in range(num_epochs_compare):
        epoch_loss = 0.0
        perm = np.random.permutation(len(X_train))
        for i in perm:
            loss = model.train_step(X_train[i], Y_train[i])
            epoch_loss += loss
        avg_loss = epoch_loss / len(X_train)
        losses.append(avg_loss)

    # Predict on test set
    preds = []
    for i in range(len(X_test)):
        pred, _ = model.forward(X_test[i])
        preds.append(pred)
    preds = np.array(preds)

    test_mse = mse_loss(preds, Y_test)
    results[name] = {'losses': losses, 'predictions': preds, 'test_mse': test_mse}

    print(f"  {name:12s}: Final train MSE = {losses[-1]:.6f}, Test MSE = {test_mse:.6f}")

print("=" * 60)
print("Done!")

In [ ]:
# ===========================================================
# Comparison plots
# ===========================================================

colors = {'Vanilla RNN': '#E67E22', 'LSTM': '#2980B9', 'GRU': '#27AE60'}

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Convergence comparison (loss curves)
for name, res in results.items():
    axes[0].plot(res['losses'], label=f"{name} (test MSE: {res['test_mse']:.5f})",
                 color=colors[name], linewidth=2)

axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Training MSE', fontsize=12)
axes[0].set_title('Convergence Comparison: RNN vs LSTM vs GRU', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Plot 2: Predictions on test set
axes[1].plot(test_x, Y_test, color='black', linewidth=2, label='Actual', alpha=0.7)

for name, res in results.items():
    axes[1].plot(test_x, res['predictions'], label=name,
                 color=colors[name], linewidth=1.5, linestyle='--')

axes[1].set_xlabel('Time step', fontsize=12)
axes[1].set_ylabel('Value', fontsize=12)
axes[1].set_title('Test Set Predictions: RNN vs LSTM vs GRU', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nComparison summary:")
print("-" * 40)
for name, res in results.items():
    print(f"  {name:12s}  Test MSE: {res['test_mse']:.6f}")
print("-" * 40)
print()
print("Typical observations:")
print("  - LSTM and GRU usually converge faster and achieve lower error.")
print("  - Vanilla RNN can struggle with the longer-range sine patterns.")
print("  - GRU is often competitive with LSTM despite being simpler.")
print("  - Results vary with random seed, hidden size, and learning rate.")

---

## Practical Tips for Sequence Tasks

Before we wrap up, here are some practical tips that apply across all sequence tasks. These are the things that make the difference between a toy example and a real-world model.

### 1. Sequence Padding

Real-world sentences have different lengths. If you want to process them in batches (for speed), you need to make them the same length. The standard approach is **padding**: add a special `<PAD>` token to the end of shorter sequences.

```
Original:         "the cat sat"     "the dog sat on the mat"
After padding:    "the cat sat <PAD> <PAD> <PAD>"   "the dog sat on the mat"
```

You then use a **mask** to tell the model to ignore the padding tokens.

### 2. Embedding Layers

In our examples we used **one-hot encoding** for words and characters. This works, but has a problem: every word is equally distant from every other word ("cat" is as far from "dog" as it is from "democracy").

In practice, you use **learned embeddings**: a dense vector for each word that captures its meaning. Words with similar meanings end up close together in this vector space. We will not implement embeddings in this notebook, but it is worth knowing about. Popular embeddings include Word2Vec, GloVe, and learned embeddings trained jointly with the model.

### 3. Gradient Clipping

We already saw this in Notebook 04, but it bears repeating: **always clip gradients when training RNNs**. The gradients in recurrent networks can explode (grow to enormous values) during backpropagation through time. A simple `np.clip(gradient, -max_val, max_val)` prevents this.

```python
# Always do this before updating weights:
np.clip(gradient, -5, 5, out=gradient)
```

### 4. Teacher Forcing

During training for sequence generation tasks, you have a choice at each time step:

- **Without teacher forcing**: feed the model's own previous prediction as the next input.
- **With teacher forcing**: feed the **correct** previous output as the next input.

Teacher forcing speeds up training dramatically because the model does not compound its own mistakes. However, at generation time the model must rely on its own predictions. A common strategy is to start with 100% teacher forcing and gradually decrease it (scheduled sampling).

### 5. Bidirectional Processing

When you have the **entire** input sequence available (classification, tagging, translation encoding), you should process it in **both** directions. A bidirectional RNN reads the sequence left-to-right AND right-to-left, giving each position context from both sides.

We will build a bidirectional RNN from scratch in the next notebook!

### Quick Reference Table

| Tip | When to Use | Why |
|-----|-------------|-----|
| Sequence padding | Different-length inputs | Enables batch processing |
| Embeddings | Text data | Better word representations than one-hot |
| Gradient clipping | Always with RNNs | Prevents exploding gradients |
| Teacher forcing | Sequence generation training | Faster, more stable training |
| Bidirectional | Full sequence available | Richer context from both directions |

---

## Summary

In this notebook we built three complete sequence applications from scratch:

| Task | Pattern | Key Idea |
|------|---------|----------|
| **Text Generation** | Many-to-Many | Predict the next character at every step, then sample to generate |
| **Sentiment Analysis** | Many-to-One | Read the whole sentence, classify using the last hidden state |
| **Time Series Prediction** | Many-to-One (sliding window) | Use a window of past values to predict the next one |

### The Key Insight

All three tasks use the **exact same building blocks** (RNN, LSTM, or GRU cells). The difference is in how we **wire them up**:

- **Text generation**: output at every step, sample from distribution
- **Sentiment analysis**: output only at the last step, sigmoid for binary classification
- **Time series**: output only at the last step, linear for regression

We also compared vanilla RNN, LSTM, and GRU on the same task and saw that gated architectures (LSTM, GRU) tend to converge faster and handle longer-range patterns better.

### What is Next?

In **Notebook 06** we will build **Bidirectional RNNs** -- models that read a sequence in both directions for richer context. This is essential for tasks like text classification and named entity recognition where you have the full input available.

---

*Great work completing Notebook 05! You now know how to apply RNNs to real sequence tasks.*